# Proyecto Integrador Módulo 3, TechCore

## Avance 2, Modelo Relacional

Simón Bedoya Montiel

Este notebook toma el dataset limpio del Avance 1 y construye a partir de él un modelo
relacional. El trabajo consiste en pasar de una tabla plana, donde toda la información
está mezclada y repetida, a un conjunto de tablas separadas por entidad, conectadas
entre sí mediante llaves.

El proceso tiene cinco partes: cargar los datos, validar su consistencia y corregir las
incongruencias encontradas, construir las tablas del modelo, verificar la integridad
referencial, y generar los reportes exploratorios y el archivo de salida.

## 1. Preparación y carga de los datos

Cargo el dataset limpio que exporté desde Power Query en el Avance 1. Este archivo ya
tiene los nombres de columna estandarizados, las ciudades, sucursales y marcas
normalizadas, los duplicados eliminados y los valores nulos tratados.

In [24]:
import pandas as pd
import numpy as np

# Cargo el dataset limpio que salio del Avance 1
df = pd.read_csv('../data/processed/ventasTransformed.csv')

print(f'Filas: {len(df)}')
print(f'Columnas: {len(df.columns)}')
df.head(3)

Filas: 30000
Columnas: 32


,VentaID,FechaVenta,HoraVenta,SucursalNombre,CiudadSucursal,VendedorNombre,ClienteNombre,GeneroCliente,EdadCliente,EmailCliente,...,SubtotalProducto2,NombreProducto3,MarcaProducto3,CantidadProducto3,PrecioUnitarioProducto3,SubtotalProducto3,DescuentoVenta,TotalVenta,AñoVenta,MesVenta
0,1,31/12/2015,05:42 a. m.,TechCore Pereira,Pereira,Amílcar Ortega-Alberto,Bienvenida Nebot-Fiol,F,37,bienvenida19@hotmail.com,...,160000000.0,Dell Latitude 7420,Dell,10.0,56000000.0,56000000.0,0,312000000,2015,12
1,2,23/03/2019,07:03 p. m.,TechCore Medellín #1,Medellín,Ana Sofía Llopis Blázquez,Teófila Bueno-Novoa,F,43,teófila35@gmail.com,...,68000000.0,NaN,NaN,NaN,NaN,NaN,0,108000000,2019,3
2,3,23/12/2018,02:32 a. m.,TechCore Medellín #2,Medellín,Juan José Porcel-Riera,Gilberto Chamorro Catalá,M,38,gilberto57@hotmail.com,...,72000000.0,HP Pavilion 15,HP,10.0,35000000.0,35000000.0,0,219000000,2018,12


## 2. Control de calidad: detección de incongruencias

Antes de construir el modelo hay que validar que los datos sean coherentes. Un modelo
relacional construido sobre datos incongruentes propaga esos errores a todo el análisis
posterior.

Revisando los precios encontré un problema que explico a continuación.

In [25]:
# Junto los tres productos de cada factura en una sola tabla para analizarlos.
# El dataset trae los productos en columnas repetidas (Producto1, Producto2, Producto3),
# asi que los apilo uno debajo del otro para poder estudiarlos como un conjunto.
partes = []
for i in [1, 2, 3]:
    p = df[[f'NombreProducto{i}', f'MarcaProducto{i}', f'PrecioUnitarioProducto{i}']].copy()
    p.columns = ['Nombre', 'Marca', 'Precio']
    partes.append(p)

productos_todos = pd.concat(partes).dropna(subset=['Nombre'])

# Cuantos precios distintos tiene cada producto? Deberia ser uno solo.
precios_por_producto = productos_todos.groupby(['Nombre', 'Marca'])['Precio'].nunique()

print(f'Productos unicos: {len(precios_por_producto)}')
print(f'Productos con MAS de un precio: {(precios_por_producto > 1).sum()}')

Productos unicos: 40
Productos con MAS de un precio: 40


### El hallazgo: precios con un cero de más

Los 40 productos tienen dos precios distintos cada uno. Al comparar esos dos precios,
la relación entre ellos es siempre la misma: uno es exactamente diez veces el otro.

Un patrón tan sistemático no es casualidad. Es un error de captura: a una parte de los
registros se les agregó un cero de más al precio.

In [26]:
# Comparo el precio mayor contra el menor de cada producto. Si la razon entre ambos
# es siempre 10, confirma que se trata de un cero de mas y no de precios legitimos
# distintos (como una promocion o un cambio de precio en el tiempo).
resumen = productos_todos.groupby(['Nombre', 'Marca'])['Precio'].agg(['min', 'max'])
resumen['razon'] = resumen['max'] / resumen['min']

print('Razones encontradas entre el precio mayor y el menor:')
print(resumen['razon'].value_counts())
print()
print('Muestra de productos afectados:')
resumen.head(6)

Razones encontradas entre el precio mayor y el menor:
razon
10.0    40
Name: count, dtype: int64

Muestra de productos afectados:


,,min,max,razon
Nombre,Marca,,,
Acer Aspire 5,Acer,2000000.0,20000000.0,10.0
Acer Nitro 5,Acer,4000000.0,40000000.0,10.0
Acer Predator Helios 300,Acer,5600000.0,56000000.0,10.0
Acer Swift 3,Acer,2600000.0,26000000.0,10.0
Apple MacBook Air M1,Apple,5500000.0,55000000.0,10.0
Apple MacBook Pro 14,Apple,8000000.0,80000000.0,10.0


La razón es exactamente 10 en los 40 productos, sin excepción. Queda confirmado que
el precio alto es el erróneo.

**Cuál es el precio correcto.** El precio menor es el válido. Basta con mirar los valores
para comprobarlo: un Apple MacBook Pro 16 cuesta alrededor de nueve millones seiscientos
mil pesos, no noventa y seis millones. Un Acer Aspire 5 de gama de entrada cuesta dos
millones, no veinte. Los precios altos son diez veces lo que deberían ser.

In [27]:
# Armo el precio correcto de cada producto: el menor de los dos.
precio_correcto = productos_todos.groupby(['Nombre', 'Marca'])['Precio'].min().reset_index()
precio_correcto.columns = ['Nombre', 'Marca', 'PrecioCorrecto']

print('Precio correcto de cada producto:')
precio_correcto.sort_values('PrecioCorrecto', ascending=False).head(8)

Precio correcto de cada producto:


,Nombre,Marca,PrecioCorrecto
24,MSI GE76 Raider,MSI,10000000.0
33,Razer Blade 17,Razer,10000000.0
6,Apple MacBook Pro 16,Apple,9600000.0
32,Razer Blade 15,Razer,8800000.0
12,Dell Alienware m15,Dell,8000000.0
5,Apple MacBook Pro 14,Apple,8000000.0
26,MSI GS66 Stealth,MSI,8000000.0
21,Lenovo Legion 5 Pro,Lenovo,7200000.0


### La segunda incongruencia: el total de la venta no cuadra

Al validar los totales encontré otro problema. El TotalVenta que trae el archivo no
coincide con la suma de los subtotales en ninguna de las 30.000 facturas.

In [28]:
# El total de una factura deberia ser la suma de sus subtotales, menos el descuento.
# Verifico si eso se cumple con los datos tal como vienen.
suma_subtotales = df[['SubtotalProducto1', 'SubtotalProducto2', 'SubtotalProducto3']].sum(axis=1)
total_esperado = suma_subtotales * (1 - df['DescuentoVenta'] / 100)

coincide = (abs(total_esperado - df['TotalVenta']) < 1)

print(f'Facturas donde el total cuadra: {coincide.sum()} de {len(df)}')
print()
print('Ejemplo de una factura que no cuadra:')
pd.DataFrame({
    'Suma subtotales': suma_subtotales.head(3),
    'Descuento': df['DescuentoVenta'].head(3),
    'Total esperado': total_esperado.head(3),
    'Total en el archivo': df['TotalVenta'].head(3)
})

Facturas donde el total cuadra: 0 de 30000

Ejemplo de una factura que no cuadra:


,Suma subtotales,Descuento,Total esperado,Total en el archivo
0,225600000.0,0,225600000.0,312000000
1,72000000.0,0,72000000.0,108000000
2,118200000.0,0,118200000.0,219000000


El total no cuadra en ninguna factura. Como los precios traen el error del cero de más
mezclado de forma inconsistente, el total del archivo se calculó sobre valores erróneos
y no es confiable.

**La decisión:** Corrijo los precios y recalculo los subtotales y el total desde cero, con
la fórmula correcta. Así los números quedan coherentes y verificables, y el modelo se
construye sobre datos que cuadran.

### La tercera incongruencia: ciudades marcadas como No especificado

En el Avance 1 rellené las ciudades vacías con el texto No especificado, siguiendo el
criterio de dejar constancia del dato faltante. Al construir el modelo descubrí que esa
decisión, aunque razonable en su momento, perdía información que sí estaba disponible.

In [29]:
# Reviso la relacion entre sucursal y ciudad. Cada sucursal deberia tener UNA ciudad.
combinaciones = df[['SucursalNombre', 'CiudadSucursal']].drop_duplicates()

print(f'Combinaciones de sucursal y ciudad: {len(combinaciones)}')
print('Pero solo hay 6 sucursales reales, asi que algo no cuadra:')
combinaciones.sort_values('SucursalNombre')

Combinaciones de sucursal y ciudad: 12
Pero solo hay 6 sucursales reales, asi que algo no cuadra:


,SucursalNombre,CiudadSucursal
22,TechCore Bogotá #1,Bogotá
50,TechCore Bogotá #1,No especificado
8,TechCore Bogotá #2,Bogotá
594,TechCore Bogotá #2,No especificado
4,TechCore Cali,Cali
622,TechCore Cali,No especificado
1,TechCore Medellín #1,Medellín
211,TechCore Medellín #1,No especificado
2,TechCore Medellín #2,Medellín
172,TechCore Medellín #2,No especificado


Cada sucursal aparece dos veces: una con su ciudad correcta y otra con No especificado.
Esas segundas filas vienen de las facturas donde la ciudad venía vacía.

**El hallazgo:** La ciudad nunca estuvo realmente perdida. Si la sucursal se llama
TechCore Bogotá #1, su ciudad es Bogotá, sin ambigüedad posible. El nombre de la sucursal
contiene la ciudad. Rellenar con No especificado descartó una información que se podía
recuperar.

**La corrección:** Deduzco la ciudad a partir del nombre de la sucursal. Esto elimina el
valor No especificado de la ciudad y deja cada sucursal con su ciudad real, que es lo
correcto para el modelo.

## 3. Corrección de las incongruencias

Aplico las correcciones sobre una copia del dataset, para conservar el original intacto.

In [30]:
# Trabajo sobre una copia para no tocar el dataset original
datos = df.copy()

# CORRECCION 1: recupero la ciudad a partir del nombre de la sucursal.
# El nombre de la sucursal contiene la ciudad (por ejemplo, TechCore Bogota #1), asi que
# la ciudad se puede deducir sin ambiguedad. Armo el mapa mirando, para cada sucursal,
# cual es la ciudad real que aparece con ella (descartando el No especificado).
mapa_ciudad = (datos[datos['CiudadSucursal'] != 'No especificado']
               .groupby('SucursalNombre')['CiudadSucursal']
               .first())

# Aplico el mapa: toda sucursal recibe su ciudad real, sin importar lo que trajera antes
datos['CiudadSucursal'] = datos['SucursalNombre'].map(mapa_ciudad)

print('Ciudades despues de la correccion:')
print(datos['CiudadSucursal'].value_counts())

# CORRECCION 2: corrijo el precio de cada producto en los tres espacios.
# El metodo: cruzo cada fila con la tabla de precios correctos, y reemplazo el precio
# por el correcto. Asi no importa si la fila traia el precio bueno o el inflado, todas
# quedan con el precio correcto.
for i in [1, 2, 3]:
    datos = datos.merge(
        precio_correcto,
        left_on=[f'NombreProducto{i}', f'MarcaProducto{i}'],
        right_on=['Nombre', 'Marca'],
        how='left'
    )
    # Reemplazo el precio por el correcto (solo donde hay producto)
    datos[f'PrecioUnitarioProducto{i}'] = datos['PrecioCorrecto']
    # Recalculo el subtotal: precio corregido por la cantidad
    datos[f'SubtotalProducto{i}'] = datos[f'PrecioUnitarioProducto{i}'] * datos[f'CantidadProducto{i}']
    # Limpio las columnas que dejo el cruce
    datos = datos.drop(columns=['Nombre', 'Marca', 'PrecioCorrecto'])

print('Precios y subtotales corregidos')

Ciudades despues de la correccion:
CiudadSucursal
Medellín    13460
Bogotá       9107
Pereira      3723
Cali         3710
Name: count, dtype: int64
Precios y subtotales corregidos


In [33]:
# CORRECCION 3: recalculo el total de cada factura con la formula correcta:
# total = suma de los subtotales, menos el descuento.
# Uso fillna(0) en la suma porque las facturas de uno o dos productos tienen nulos
# en los espacios que no usaron, y para SUMAR esos nulos valen cero.
suma_subs = datos[['SubtotalProducto1', 'SubtotalProducto2', 'SubtotalProducto3']].fillna(0).sum(axis=1)
datos['TotalVenta'] = suma_subs * (1 - datos['DescuentoVenta'] / 100)

# Verifico que ahora si cuadre
verificacion = abs(suma_subs * (1 - datos['DescuentoVenta']/100) - datos['TotalVenta']) < 1
print(f'Facturas donde el total cuadra ahora: {verificacion.sum()} de {len(datos)}')

Facturas donde el total cuadra ahora: 30000 de 30000


## 4. Diseño del modelo relacional

Con los datos ya coherentes, construyo el modelo. La idea de fondo es separar la
información por entidad, para que cada dato viva en un solo lugar.

El problema de la tabla plana es la redundancia: si un cliente compró diez veces, sus
datos están repetidos diez veces. Si un producto aparece en mil facturas, su nombre y
marca están escritos mil veces. El modelo relacional resuelve eso.

### Las entidades del modelo

Identifiqué siete entidades, cada una con su tabla:

| Tabla | Qué guarda | Llave primaria |
|---|---|---|
| **Ciudades** | Las ciudades donde opera TechCore | CiudadID |
| **Sucursales** | Las tiendas físicas | SucursalID |
| **Vendedores** | El personal de ventas | VendedorID |
| **Clientes** | Los compradores | ClienteID |
| **Productos** | El catálogo de productos | ProductoID |
| **Facturas** | Cada venta realizada | FacturaID |
| **DetalleFacturas** | Cada producto de cada venta | DetalleID |

### La tabla clave: DetalleFacturas

Esta es la transformación más importante del avance. Hoy los tres productos de una
factura están en columnas repetidas dentro de la misma fila. En un modelo relacional eso
no se hace así.

En su lugar, cada producto de cada factura se convierte en una fila del detalle. Si una
factura vendió tres productos, genera tres filas. Este paso, que consiste en pasar de
columnas a filas, es lo que permite que el modelo maneje cualquier cantidad de productos
por factura sin cambiar su estructura.

### Tabla Ciudades

Las ciudades donde TechCore tiene sucursales. Es la tabla más simple y sirve de
referencia para las sucursales.

In [34]:
# Cada ciudad unica recibe su propio identificador
ciudades = pd.DataFrame({
    'NombreCiudad': sorted(datos['CiudadSucursal'].unique())
})
ciudades.insert(0, 'CiudadID', range(1, len(ciudades) + 1))

print(f'Ciudades: {len(ciudades)}')
ciudades

Ciudades: 4


,CiudadID,NombreCiudad
0,1,Bogotá
1,2,Cali
2,3,Medellín
3,4,Pereira


### Tabla Sucursales

Las tiendas físicas. Cada sucursal pertenece a una ciudad, así que lleva la llave foránea
CiudadID que apunta a la tabla de Ciudades.

In [35]:
# Cada sucursal con su ciudad. Uso drop_duplicates para quedarme con las sucursales
# unicas, ya que en el dataset cada una aparece repetida en miles de facturas.
sucursales = datos[['SucursalNombre', 'CiudadSucursal']].drop_duplicates().reset_index(drop=True)
sucursales.columns = ['NombreSucursal', 'NombreCiudad']

# Traigo el CiudadID cruzando con la tabla de ciudades. Esta es la llave foranea.
sucursales = sucursales.merge(ciudades, on='NombreCiudad', how='left')
sucursales = sucursales[['NombreSucursal', 'CiudadID']]
sucursales.insert(0, 'SucursalID', range(1, len(sucursales) + 1))

print(f'Sucursales: {len(sucursales)}')
sucursales

Sucursales: 6


,SucursalID,NombreSucursal,CiudadID
0,1,TechCore Pereira,4
1,2,TechCore Medellín #1,3
2,3,TechCore Medellín #2,3
3,4,TechCore Cali,2
4,5,TechCore Bogotá #2,1
5,6,TechCore Bogotá #1,1


### Tabla Vendedores

El personal de ventas. Se identifica por su nombre, que es único en el dataset.

In [36]:
vendedores = pd.DataFrame({
    'NombreVendedor': sorted(datos['VendedorNombre'].unique())
})
vendedores.insert(0, 'VendedorID', range(1, len(vendedores) + 1))

print(f'Vendedores: {len(vendedores)}')
vendedores.head()

Vendedores: 30


,VendedorID,NombreVendedor
0,1,Almudena Adelaida Lledó Garrido
1,2,Amílcar Ortega-Alberto
2,3,Ana Sofía Llopis Blázquez
3,4,Anacleto Morera Segovia
4,5,Armida Azorin Plaza


### Tabla Clientes

Aquí hay una decisión de diseño que vale la pena explicar.

**El problema.** El dataset no trae un identificador de cliente. Lo natural sería usar el
correo, pero al revisarlo encontré que 679 correos están asociados a personas distintas
(por ejemplo, el mismo correo aparece con dos nombres diferentes). Esto ocurre porque los
correos se generaron a partir del nombre de pila más un número, así que dos personas con
el mismo primer nombre pueden compartir correo.

**La decisión:** Identifico a cada cliente por la combinación completa de sus datos: nombre,
género, edad, correo, teléfono y dirección. Si dos filas coinciden en todo eso, es la
misma persona. Es más robusto que confiar en un solo campo.

In [37]:
# Cada cliente unico se define por la combinacion de todos sus datos, no por un solo
# campo. Ver la explicacion de arriba: el correo no sirve como identificador porque
# hay correos compartidos entre personas distintas.
cols_cliente = ['ClienteNombre', 'GeneroCliente', 'EdadCliente',
                'EmailCliente', 'TelefonoCliente', 'DireccionCliente']

clientes = datos[cols_cliente].drop_duplicates().reset_index(drop=True)
clientes.columns = ['NombreCliente', 'Genero', 'Edad', 'Email', 'Telefono', 'Direccion']
clientes.insert(0, 'ClienteID', range(1, len(clientes) + 1))

print(f'Clientes unicos: {len(clientes)}')
clientes.head()

Clientes unicos: 17453


,ClienteID,NombreCliente,Genero,Edad,Email,Telefono,Direccion
0,1,Bienvenida Nebot-Fiol,F,37,bienvenida19@hotmail.com,+34806548767,cll 52 #32-98
1,2,Teófila Bueno-Novoa,F,43,teófila35@gmail.com,+34 947255990,cll 96 #81-94
2,3,Gilberto Chamorro Catalá,M,38,gilberto57@hotmail.com,+34 978 810 249,cra 28 #79-85
3,4,Máximo Coronado Huerta,M,30,máximo65@hotmail.com,+34 825429634,cll 5 #89-26
4,5,Yago Oliver,M,51,yago38@yahoo.com,+34988285275,cra 32 #97-86


### Tabla Productos

El catálogo. Cada producto se identifica por la combinación de su nombre y su marca,
porque un mismo nombre podría existir en varias marcas. Lleva el precio ya corregido.

In [38]:
# El catalogo de productos, con su precio corregido. Cada producto es unico por la
# combinacion de nombre y marca.
productos = precio_correcto.copy()
productos.columns = ['NombreProducto', 'Marca', 'Precio']
productos = productos.sort_values(['Marca', 'NombreProducto']).reset_index(drop=True)
productos.insert(0, 'ProductoID', range(1, len(productos) + 1))

print(f'Productos: {len(productos)}')
productos.head(8)

Productos: 40


,ProductoID,NombreProducto,Marca,Precio
0,1,Acer Aspire 5,Acer,2000000.0
1,2,Acer Nitro 5,Acer,4000000.0
2,3,Acer Predator Helios 300,Acer,5600000.0
3,4,Acer Swift 3,Acer,2600000.0
4,5,Apple MacBook Air M1,Apple,5500000.0
5,6,Apple MacBook Pro 14,Apple,8000000.0
6,7,Apple MacBook Pro 16,Apple,9600000.0
7,8,Apple iMac 24,Apple,6400000.0


### Tabla Facturas

Cada venta realizada. Es la tabla central del modelo, y lleva tres llaves foráneas que
la conectan con la sucursal donde se hizo la venta, el cliente que compró y el vendedor
que atendió.

La factura no guarda los productos. Esos van en la tabla de detalle, porque una
factura puede tener varios productos y una tabla no puede tener un número variable de
columnas.

In [39]:
# La tabla de facturas. Parto del dataset y le traigo los identificadores de sucursal,
# cliente y vendedor mediante cruces. Esas son las llaves foraneas que conectan la
# factura con las demas entidades.
facturas = datos[['VentaID', 'FechaVenta', 'HoraVenta', 'SucursalNombre',
                  'VendedorNombre', 'MetodoPago', 'DescuentoVenta', 'TotalVenta',
                  'AñoVenta', 'MesVenta'] + cols_cliente].copy()

# Traigo el SucursalID. Uso un mapa en vez de un merge para evitar cualquier riesgo
# de duplicar filas si la tabla de referencia tuviera repetidos.
mapa_sucursal = sucursales.set_index('NombreSucursal')['SucursalID']
facturas['SucursalID'] = facturas['SucursalNombre'].map(mapa_sucursal)

# Traigo el VendedorID, tambien con un mapa
mapa_vendedor = vendedores.set_index('NombreVendedor')['VendedorID']
facturas['VendedorID'] = facturas['VendedorNombre'].map(mapa_vendedor)

# Traigo el ClienteID. El cruce va por los seis campos que identifican al cliente,
# porque asi fue como lo defini.
clientes_para_cruce = clientes.copy()
clientes_para_cruce.columns = ['ClienteID'] + cols_cliente
facturas = facturas.merge(clientes_para_cruce, on=cols_cliente, how='left')

# Me quedo solo con las columnas que van en la tabla de facturas
facturas = facturas[['VentaID', 'FechaVenta', 'HoraVenta', 'SucursalID', 'ClienteID',
                     'VendedorID', 'MetodoPago', 'DescuentoVenta', 'TotalVenta',
                     'AñoVenta', 'MesVenta']]
facturas = facturas.rename(columns={'VentaID': 'FacturaID', 'FechaVenta': 'Fecha',
                                     'HoraVenta': 'Hora', 'DescuentoVenta': 'Descuento',
                                     'AñoVenta': 'Año', 'MesVenta': 'Mes'})

print(f'Facturas: {len(facturas)}')
facturas.head()

Facturas: 30000


,FacturaID,Fecha,Hora,SucursalID,ClienteID,VendedorID,MetodoPago,Descuento,TotalVenta,Año,Mes
0,1,31/12/2015,05:42 a. m.,1,1,2,Tarjeta Crédito,0,225600000.0,2015,12
1,2,23/03/2019,07:03 p. m.,2,2,3,Billetera Digital,0,72000000.0,2019,3
2,3,23/12/2018,02:32 a. m.,3,3,22,Tarjeta Débito,0,118200000.0,2018,12
3,4,14/11/2015,12:37 a. m.,3,4,22,Billetera Digital,0,92800000.0,2015,11
4,5,29/11/2016,10:34 a. m.,4,5,20,Tarjeta Crédito,0,92200000.0,2016,11


### Tabla DetalleFacturas

La transformación más importante. Aquí paso los productos de columnas a filas.

Cada producto de cada factura se convierte en una fila del detalle. Una factura con tres
productos genera tres filas. Una con un solo producto genera una.

Este paso es el que hace que el modelo pueda manejar cualquier cantidad de productos por
factura, sin tener que agregar columnas. Es la esencia del modelo relacional.

In [40]:
# Paso los productos de columnas a filas. Recorro los tres espacios de producto y,
# por cada uno, armo un bloque con las facturas que SI tienen producto en ese espacio.
# Las facturas de un solo producto solo aportan al primer bloque, las de dos aportan a
# los dos primeros, y asi.
bloques = []

for i in [1, 2, 3]:
    bloque = datos[['VentaID', f'NombreProducto{i}', f'MarcaProducto{i}',
                    f'CantidadProducto{i}', f'PrecioUnitarioProducto{i}',
                    f'SubtotalProducto{i}', 'DescuentoVenta']].copy()
    bloque.columns = ['FacturaID', 'NombreProducto', 'Marca', 'Cantidad',
                      'PrecioUnitario', 'Subtotal', 'Descuento']
    # Me quedo solo con las filas que SI tienen producto en este espacio.
    # Las que no lo tienen simplemente no aportan una fila al detalle, que es lo
    # correcto: si la factura vendio un solo producto, genera una sola fila.
    bloque = bloque.dropna(subset=['NombreProducto'])
    bloques.append(bloque)

detalle = pd.concat(bloques, ignore_index=True)

# Traigo el ProductoID cruzando por nombre y marca, que es como identifico un producto
detalle = detalle.merge(
    productos[['ProductoID', 'NombreProducto', 'Marca']],
    on=['NombreProducto', 'Marca'], how='left'
)

# Ordeno por factura para que el detalle quede legible, y le pongo su llave primaria
detalle = detalle.sort_values('FacturaID').reset_index(drop=True)
detalle.insert(0, 'DetalleID', range(1, len(detalle) + 1))

detalle = detalle[['DetalleID', 'FacturaID', 'ProductoID', 'Cantidad',
                   'PrecioUnitario', 'Subtotal', 'Descuento']]
detalle['Cantidad'] = detalle['Cantidad'].astype(int)

print(f'Lineas de detalle: {len(detalle)}')
print(f'Facturas: {len(facturas)}')
print(f'Promedio de productos por factura: {len(detalle) / len(facturas):.2f}')
detalle.head(8)

Lineas de detalle: 60059
Facturas: 30000
Promedio de productos por factura: 2.00


,DetalleID,FacturaID,ProductoID,Cantidad,PrecioUnitario,Subtotal,Descuento
0,1,1,7,1,9600000.0,9600000.0,0
1,2,1,27,20,8000000.0,160000000.0,0
2,3,1,15,10,5600000.0,56000000.0,0
3,4,2,23,10,6800000.0,68000000.0,0
4,5,2,2,1,4000000.0,4000000.0,0
5,6,3,22,10,7200000.0,72000000.0,0
6,7,3,19,10,3500000.0,35000000.0,0
7,8,3,15,2,5600000.0,11200000.0,0


## 5. Validación de la integridad referencial

La integridad referencial significa que toda llave foránea apunta a un registro que
existe de verdad. Si el detalle de una factura apunta a una factura que no existe, o a
un producto que no está en el catálogo, el modelo está roto.

Estos registros sin padre se llaman **huérfanos**, y el objetivo es que no haya ninguno.

In [41]:
# Cada validacion cuenta los registros que apuntan a algo que NO existe.
# Lo esperado es que todas den cero.
print('VALIDACION DE INTEGRIDAD REFERENCIAL')
print('=' * 50)

validaciones = {
    'DetalleFacturas -> Facturas (FacturaID)':
        (~detalle['FacturaID'].isin(facturas['FacturaID'])).sum(),
    'DetalleFacturas -> Productos (ProductoID)':
        (~detalle['ProductoID'].isin(productos['ProductoID'])).sum(),
    'Facturas -> Sucursales (SucursalID)':
        (~facturas['SucursalID'].isin(sucursales['SucursalID'])).sum(),
    'Facturas -> Clientes (ClienteID)':
        (~facturas['ClienteID'].isin(clientes['ClienteID'])).sum(),
    'Facturas -> Vendedores (VendedorID)':
        (~facturas['VendedorID'].isin(vendedores['VendedorID'])).sum(),
    'Sucursales -> Ciudades (CiudadID)':
        (~sucursales['CiudadID'].isin(ciudades['CiudadID'])).sum(),
}

todo_bien = True
for nombre, huerfanos in validaciones.items():
    estado = 'OK' if huerfanos == 0 else f'ERROR: {huerfanos} huerfanos'
    print(f'  {nombre}: {estado}')
    if huerfanos > 0:
        todo_bien = False

print()
print('Integridad referencial:', 'CORRECTA' if todo_bien else 'CON PROBLEMAS')

VALIDACION DE INTEGRIDAD REFERENCIAL
  DetalleFacturas -> Facturas (FacturaID): OK
  DetalleFacturas -> Productos (ProductoID): OK
  Facturas -> Sucursales (SucursalID): OK
  Facturas -> Clientes (ClienteID): OK
  Facturas -> Vendedores (VendedorID): OK
  Sucursales -> Ciudades (CiudadID): OK

Integridad referencial: CORRECTA


In [42]:
# Tambien valido que las llaves primarias sean realmente unicas. Una llave primaria
# repetida romperia el modelo, porque dejaria de identificar de forma unica a cada fila.
print('VALIDACION DE LLAVES PRIMARIAS (deben ser unicas)')
print('=' * 50)

tablas = {
    'Ciudades': (ciudades, 'CiudadID'),
    'Sucursales': (sucursales, 'SucursalID'),
    'Vendedores': (vendedores, 'VendedorID'),
    'Clientes': (clientes, 'ClienteID'),
    'Productos': (productos, 'ProductoID'),
    'Facturas': (facturas, 'FacturaID'),
    'DetalleFacturas': (detalle, 'DetalleID'),
}

for nombre, (tabla, llave) in tablas.items():
    duplicados = tabla[llave].duplicated().sum()
    nulos = tabla[llave].isna().sum()
    estado = 'OK' if (duplicados == 0 and nulos == 0) else f'ERROR: {duplicados} dup, {nulos} nulos'
    print(f'  {nombre} ({llave}): {estado}  [{len(tabla)} filas]')

VALIDACION DE LLAVES PRIMARIAS (deben ser unicas)
  Ciudades (CiudadID): OK  [4 filas]
  Sucursales (SucursalID): OK  [6 filas]
  Vendedores (VendedorID): OK  [30 filas]
  Clientes (ClienteID): OK  [17453 filas]
  Productos (ProductoID): OK  [40 filas]
  Facturas (FacturaID): OK  [30000 filas]
  DetalleFacturas (DetalleID): OK  [60059 filas]


In [43]:
# Validacion de coherencia: la suma de los subtotales del detalle de cada factura,
# menos el descuento, debe dar el total de la factura. Esto comprueba que la
# transformacion de columnas a filas no perdio ni duplico informacion.
suma_por_factura = detalle.groupby('FacturaID')['Subtotal'].sum().reset_index()
suma_por_factura.columns = ['FacturaID', 'SumaDetalle']

chequeo = facturas[['FacturaID', 'Descuento', 'TotalVenta']].merge(suma_por_factura, on='FacturaID')
chequeo['TotalCalculado'] = chequeo['SumaDetalle'] * (1 - chequeo['Descuento'] / 100)
chequeo['Cuadra'] = abs(chequeo['TotalCalculado'] - chequeo['TotalVenta']) < 1

print('VALIDACION DE COHERENCIA DE TOTALES')
print('=' * 50)
print(f'  Facturas donde el total cuadra con su detalle: {chequeo["Cuadra"].sum()} de {len(chequeo)}')
print(f'  Facturas con descuadre: {(~chequeo["Cuadra"]).sum()}')

VALIDACION DE COHERENCIA DE TOTALES
  Facturas donde el total cuadra con su detalle: 30000 de 30000
  Facturas con descuadre: 0


## 6. Reportes exploratorios

La consigna pide generar reportes rápidos para confirmar que el modelo quedó bien
estructurado. Si los números tienen sentido, el modelo está bien armado.

### Total de ventas por marca

In [44]:
# Cruzo el detalle con los productos para saber la marca de cada linea vendida,
# y sumo las ventas de cada marca.
ventas_marca = detalle.merge(productos[['ProductoID', 'Marca']], on='ProductoID')
ventas_marca = ventas_marca.groupby('Marca').agg(
    TotalVendido=('Subtotal', 'sum'),
    UnidadesVendidas=('Cantidad', 'sum'),
    LineasDeVenta=('DetalleID', 'count')
).sort_values('TotalVendido', ascending=False).reset_index()

# Formateo el total en millones para que se lea mejor
ventas_marca['TotalVendido_Millones'] = (ventas_marca['TotalVendido'] / 1_000_000).round(1)

print('TOTAL DE VENTAS POR MARCA')
ventas_marca[['Marca', 'TotalVendido_Millones', 'UnidadesVendidas', 'LineasDeVenta']]

TOTAL DE VENTAS POR MARCA


,Marca,TotalVendido_Millones,UnidadesVendidas,LineasDeVenta
0,Lenovo,656165.6,120913,14468
1,HP,529810.0,108968,13243
2,Dell,476146.6,89008,10821
3,Apple,422534.4,58456,7176
4,Asus,182941.4,44871,5418
5,Acer,121788.8,39497,4800
6,Samsung,67596.0,14093,1770
7,MSI,47754.4,7584,922
8,Microsoft,45193.6,9475,1133
9,Razer,22274.8,2732,308


### Top 10 de productos más vendidos

In [45]:
# Los productos con mas unidades vendidas. Cruzo el detalle con el catalogo para
# traer el nombre y la marca de cada producto.
top_productos = detalle.merge(
    productos[['ProductoID', 'NombreProducto', 'Marca', 'Precio']], on='ProductoID'
)
top_productos = top_productos.groupby(['NombreProducto', 'Marca', 'Precio']).agg(
    UnidadesVendidas=('Cantidad', 'sum'),
    TotalVendido=('Subtotal', 'sum')
).sort_values('UnidadesVendidas', ascending=False).head(10).reset_index()

top_productos['TotalVendido_Millones'] = (top_productos['TotalVendido'] / 1_000_000).round(1)

print('TOP 10 PRODUCTOS MAS VENDIDOS')
top_productos[['NombreProducto', 'Marca', 'UnidadesVendidas', 'TotalVendido_Millones']]

TOP 10 PRODUCTOS MAS VENDIDOS


,NombreProducto,Marca,UnidadesVendidas,TotalVendido_Millones
0,HP Spectre x360,HP,41350,215020.0
1,Lenovo ThinkPad X1 Carbon,Lenovo,32724,222523.2
2,Lenovo Legion 5 Pro,Lenovo,32033,230637.6
3,Lenovo Yoga 7i,Lenovo,28605,125862.0
4,HP Omen 16,HP,27781,166686.0
5,Lenovo IdeaPad 5,Lenovo,27551,77142.8
6,Dell Latitude 7420,Dell,24910,139496.0
7,Dell XPS 13,Dell,24812,119097.6
8,HP Pavilion 15,HP,22488,78708.0
9,Dell Alienware m15,Dell,19939,159512.0


### Reportes adicionales

Dos reportes más que ayudan a confirmar que el modelo responde bien a preguntas del
negocio, cruzando distintas tablas.

In [46]:
# Ventas por ciudad. Este reporte cruza cuatro tablas: detalle, facturas, sucursales
# y ciudades. Que funcione confirma que las relaciones del modelo estan bien armadas.
ventas_ciudad = (detalle
    .merge(facturas[['FacturaID', 'SucursalID']], on='FacturaID')
    .merge(sucursales[['SucursalID', 'CiudadID']], on='SucursalID')
    .merge(ciudades, on='CiudadID')
    .groupby('NombreCiudad').agg(
        TotalVendido=('Subtotal', 'sum'),
        Facturas=('FacturaID', 'nunique')
    ).sort_values('TotalVendido', ascending=False).reset_index()
)
ventas_ciudad['TotalVendido_Millones'] = (ventas_ciudad['TotalVendido'] / 1_000_000).round(1)

print('VENTAS POR CIUDAD')
ventas_ciudad[['NombreCiudad', 'TotalVendido_Millones', 'Facturas']]

VENTAS POR CIUDAD


,NombreCiudad,TotalVendido_Millones,Facturas
0,Medellín,1146716.3,13460
1,Bogotá,791614.8,9107
2,Pereira,318000.0,3723
3,Cali,315874.5,3710


In [47]:
# Ventas por año. Sirve para ver la evolucion del negocio en el tiempo.
ventas_anio = facturas.groupby('Año').agg(
    TotalVendido=('TotalVenta', 'sum'),
    Facturas=('FacturaID', 'count')
).reset_index()
ventas_anio['TotalVendido_Millones'] = (ventas_anio['TotalVendido'] / 1_000_000).round(1)

print('VENTAS POR AÑO')
ventas_anio[['Año', 'TotalVendido_Millones', 'Facturas']]

VENTAS POR AÑO


,Año,TotalVendido_Millones,Facturas
0,2014,74469.8,832
1,2015,224669.2,2640
2,2016,224547.8,2638
3,2017,237954.5,2824
4,2018,234275.8,2755
5,2019,233146.5,2757
6,2020,226482.3,2686
7,2021,229033.4,2740
8,2022,231623.4,2765
9,2023,235794.6,2778


## 7. Generación del archivo de salida

Guardo las siete tablas en un archivo de Excel, cada una en su propia hoja. Este archivo
es el que alimentará el modelo y las visualizaciones de Power BI en el Avance 3.

In [48]:
# Cada tabla va en su propia hoja del Excel. Power BI leera cada hoja como una tabla
# distinta, y ahi se reconstruiran las relaciones del modelo.
with pd.ExcelWriter('../data/model/modeloVentas.xlsx', engine='openpyxl') as writer:
    ciudades.to_excel(writer, sheet_name='Ciudades', index=False)
    sucursales.to_excel(writer, sheet_name='Sucursales', index=False)
    vendedores.to_excel(writer, sheet_name='Vendedores', index=False)
    clientes.to_excel(writer, sheet_name='Clientes', index=False)
    productos.to_excel(writer, sheet_name='Productos', index=False)
    facturas.to_excel(writer, sheet_name='Facturas', index=False)
    detalle.to_excel(writer, sheet_name='DetalleFacturas', index=False)

print('Archivo modeloVentas.xlsx generado')
print()
print('RESUMEN DEL MODELO')
print('=' * 50)
for nombre, (tabla, llave) in tablas.items():
    print(f'  {nombre:18} {len(tabla):>7} filas')

Archivo modeloVentas.xlsx generado

RESUMEN DEL MODELO
  Ciudades                 4 filas
  Sucursales               6 filas
  Vendedores              30 filas
  Clientes             17453 filas
  Productos               40 filas
  Facturas             30000 filas
  DetalleFacturas      60059 filas


## 8. Resumen del avance

El modelo relacional quedó construido y validado. Se pasó de una tabla plana de 30.000
filas y 32 columnas, con información repetida, a siete tablas conectadas por llaves, donde
cada dato vive en un solo lugar.

**Las correcciones aplicadas:** El control de calidad detectó dos incongruencias que se
corrigieron: los precios de los 40 productos venían duplicados, con una versión inflada
diez veces por un cero de más, y el total de las facturas no cuadraba con la suma de sus
subtotales en ninguna fila. Los precios se unificaron al valor correcto, y los subtotales
y totales se recalcularon con la fórmula correcta.

**La transformación clave:** Los productos pasaron de estar en columnas repetidas a ser filas
de la tabla de detalle, lo que permite que el modelo maneje cualquier cantidad de productos
por factura sin cambiar su estructura.

**La validación:** Todas las llaves foráneas apuntan a registros que existen, no hay huérfanos,
las llaves primarias son únicas, y los totales de las facturas cuadran con la suma de sus
detalles.

El archivo modeloVentas.xlsx queda listo para alimentar el modelo y el dashboard de
Power BI en el Avance 3.